<a href="https://colab.research.google.com/github/martinhdezpacheco/tfg-scraping-madrid/blob/main/extraer_inmueble_tecnocasa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4

In [ ]:
'''================================================
FASE 1 (TECNOCASA) : Extracción de URLs de Tecnocasa
================================================'''

# ATENCIÓN!!!!!! Eliminar antes los archivos de Colab

import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os

In [ ]:
"""EJEMPLO DE EXTRACCIÓN DE DATOS DE UNA FICHA DE INMUEBLE EN TECNOCASA"""

def extraer_datos_inmueble(url):
    """
    Descarga y extrae los datos de una única ficha de inmueble de Tecnocasa.

    Parámetro:
        url (str): URL completa de la ficha, ej:
                   "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"

    Devuelve:
        dict con los campos limpios, o None si algo falló.
    """

    # --- PASO 1: Descargar el HTML crudo ---
    # Añadimos un "User-Agent" que simula un navegador real. En
    # books.toscrape.com no hacía falta porque es una web de práctica
    # sin ninguna protección, pero en una web real como Tecnocasa,
    # si no lo mandamos, el servidor puede rechazar la petición o
    # devolver una versión distinta de la página (o directamente un error).
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        respuesta = requests.get(url, headers=headers, timeout=10)
        respuesta.raise_for_status()  # lanza un error si la petición falló (404, 500...)
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar {url}: {e}")
        return None

    # --- PASO 2: Parsear el HTML con BeautifulSoup ---
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # --- PASO 3: Encontrar la etiqueta <estate-show-v2> ---
    # Aunque no es una etiqueta HTML "de verdad" (es un componente Vue.js
    # personalizado), BeautifulSoup la trata exactamente igual que
    # cualquier otra etiqueta (<div>, <span>, etc.) porque para
    # BeautifulSoup, cualquier palabra entre < > es una etiqueta válida.
    tag_estate = soup.find("estate-show-v2")

    if tag_estate is None:
        print(f"No se encontró <estate-show-v2> en {url}")
        return None

    # --- PASO 4: Extraer el atributo ":estate" ---
    # Este atributo empieza por ":" (es la sintaxis de Vue.js para un
    # "binding dinámico"). Se accede igual que a cualquier otro atributo,
    # con .get("nombre_del_atributo").
    json_crudo = tag_estate.get(":estate")

    if json_crudo is None:
        print(f"No se encontró el atributo :estate en {url}")
        return None

    # --- PASO 5: "Desescapar" el texto ---
    # El HTML convierte las comillas dobles en &quot; para poder meter
    # un JSON entero dentro de un atributo HTML sin romper la sintaxis.
    # html.unescape() deshace esa conversión (&quot; -> ", &amp; -> &, etc.)
    json_texto = html.unescape(json_crudo)

    # --- PASO 6: Convertir el texto en un diccionario de Python ---
    try:
        datos_completos = json.loads(json_texto)
    except json.JSONDecodeError as e:
        print(f"Error al parsear JSON en {url}: {e}")
        return None

    # --- PASO 7: Quedarnos solo con los campos que nos interesan ---
    # datos_completos es un diccionario ENORME (incluye fotos, colegios
    # cercanos, farmacias, datos de la agencia, hipoteca...). Nosotros
    # solo necesitamos los campos relevantes para el dataset de precios.
    #
    # Usamos .get() en vez de [] porque .get() no da error si la clave
    # no existe (devuelve None) - útil porque no todas las fichas tienen
    # rellenos todos los campos.
    # Nota sobre "features.elevator" y campos similares (garden, concierge...):
    # cuando el JSON trae "" (cadena vacía) NO sabemos con certeza si significa
    # "no tiene" o "no se especificó" en el anuncio. Por seguridad, tratamos
    # cualquier valor vacío como dato AUSENTE (None), no como "no" - así evitamos
    # meter un sesgo falso en el modelo. Más adelante, con más fichas descargadas,
    # se puede revisar si esto se puede refinar comparando con el texto libre
    # de la descripción.
    inmueble = {
        "id": datos_completos.get("id"),
        "detail_url": datos_completos.get("detail_url"),
        "tipo": datos_completos.get("type", {}).get("title"),
        "distrito": datos_completos.get("district", {}).get("title"),
        "barrio": datos_completos.get("quarter"),
        "precio": datos_completos.get("numeric_price"),                       # ej: 1399000
        "m2": datos_completos.get("numeric_surface"),                         # ej: "160.00"
        "habitaciones": datos_completos.get("rooms"),                         # ej: "3 dorm." (a limpiar con regex)
        "banos": datos_completos.get("bathrooms"),                            # ej: "3 baños" (a limpiar con regex)
        "planta": datos_completos.get("features", {}).get("floor"),
        "anio_construccion": datos_completos.get("features", {}).get("build_year"),
        "anio_reforma": datos_completos.get("features", {}).get("renovation_year"),
        "categoria": datos_completos.get("features", {}).get("category"),
        "ascensor": datos_completos.get("features", {}).get("elevator"),
        "balcones": datos_completos.get("features", {}).get("balconies"),
        "terrazas": datos_completos.get("features", {}).get("terraces"),
        "jardin": datos_completos.get("features", {}).get("garden"),
        "calefaccion": datos_completos.get("features", {}).get("heating"),
        "clase_energetica": datos_completos.get("energy_data", {}).get("class"),
        # points_of_interest es una estructura anidada (colegios, farmacias,
        # bares... cada uno con nombre y distancia). No cabe en una sola
        # celda de forma "limpia", así que la guardamos como texto JSON
        # completo por ahora, para procesarla aparte más adelante.
        "points_of_interest": json.dumps(datos_completos.get("points_of_interest"), ensure_ascii=False),
        "title": datos_completos.get("title"),
        "description": datos_completos.get("description"),
    }

    # Normalizamos: cualquier campo que sea cadena vacía ("") lo convertimos
    # en None, para que al guardar en CSV quede en blanco de verdad, en vez
    # de aparecer como texto vacío o como la palabra "None".
    for clave, valor in inmueble.items():
        if valor == "":
            inmueble[clave] = None

    return inmueble


# --- PRUEBA: extraemos un único piso para comprobar que funciona ---
if __name__ == "__main__":
    url_prueba = "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"
    resultado = extraer_datos_inmueble(url_prueba)

    if resultado:
        print("Datos extraídos correctamente:\n")
        for campo, valor in resultado.items():
            print(f"{campo}: {valor}")
    else:
        print("No se pudieron extraer los datos.")

Datos extraídos correctamente:

id: 664738
detail_url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html
tipo: Piso
distrito: Las Letras Y Cortes
barrio: Huertas - Cortes
precio: 1399000
m2: 160.00
habitaciones: 3 dorm.
banos: 3 baños
planta: 3 (planta ático)
anio_construccion: 1880
anio_reforma: 2002
categoria: Media
ascensor: None
balcones: None
terrazas: None
jardin: None
calefaccion: centralizada (Radiadores)
clase_energetica: e
points_of_interest: {"public_transport": [{"name": "Antón Martín", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "200 m"}, {"name": "Lavapiés", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "550 m"}, {"name": "Sol", "class": "railway", "subclass": "station", "icon": "station", "distance": "770 m"}, {"name": "Madrid-Puerta de Atocha", "class": "railway", "subclass": "station", "icon": "station", "distance": "930 m"}, {"name": "Atocha - Costanilla Desamparados", "class": "bus", "subclass": "bus_s

In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 0: SUBIR EL CSV DEL DÍA ANTERIOR (si existe)
==================================================================="""

from google.colab import files

print("Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.")
subido = files.upload()

print("Bloque 0 completado.")

Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.


Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Bloque 0 completado.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 1: EXTRACCIÓN DE TODAS LAS URLs (estado actual de Tecnocasa)
==================================================================="""

todas_las_urls = []
pagina = 1
ids_pagina_1 = set()

while True:
    if pagina == 1:
        url_listado = "https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html"
    else:
        url_listado = f"https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html/pag-{pagina}"

    respuesta = requests.get(url_listado, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)

    if respuesta.status_code != 200:
        print(f"Página {pagina} no disponible (status {respuesta.status_code}). Fin.")
        break

    soup = BeautifulSoup(respuesta.text, "html.parser")
    tag_estates = soup.find("estates-index")

    if tag_estates is None:
        print(f"No se encontró <estates-index> en la página {pagina}. Fin.")
        break

    json_texto = html.unescape(tag_estates.get(":estates"))
    lista_anuncios = json.loads(json_texto)

    if len(lista_anuncios) == 0:
        print(f"Página {pagina} sin anuncios. Fin.")
        break

    ids_actuales = {anuncio.get("id") for anuncio in lista_anuncios}

    if pagina == 1:
        ids_pagina_1 = ids_actuales
    else:
        coincidencias = len(ids_actuales & ids_pagina_1)
        if coincidencias >= len(ids_actuales) / 2:
            print(f"Página {pagina} parece ser fallback. Fin real del listado.")
            break

    for anuncio in lista_anuncios:
        url_detalle = anuncio.get("detail_url")
        if url_detalle:
            todas_las_urls.append(url_detalle)

    print(f"Página {pagina}: {len(lista_anuncios)} anuncios encontrados (acumulado: {len(todas_las_urls)})")

    pagina += 1
    time.sleep(1)

print(f"Bloque 1 completado. Total de URLs en Tecnocasa hoy: {len(todas_las_urls)}")

Página 1: 15 anuncios encontrados (acumulado: 15)
Página 2: 15 anuncios encontrados (acumulado: 30)
Página 3: 15 anuncios encontrados (acumulado: 45)
Página 4: 15 anuncios encontrados (acumulado: 60)
Página 5: 15 anuncios encontrados (acumulado: 75)
Página 6: 15 anuncios encontrados (acumulado: 90)
Página 7: 15 anuncios encontrados (acumulado: 105)
Página 8: 15 anuncios encontrados (acumulado: 120)
Página 9: 15 anuncios encontrados (acumulado: 135)
Página 10: 15 anuncios encontrados (acumulado: 150)
Página 11: 15 anuncios encontrados (acumulado: 165)
Página 12: 15 anuncios encontrados (acumulado: 180)
Página 13: 15 anuncios encontrados (acumulado: 195)
Página 14: 15 anuncios encontrados (acumulado: 210)
Página 15: 15 anuncios encontrados (acumulado: 225)
Página 16: 15 anuncios encontrados (acumulado: 240)
Página 17: 15 anuncios encontrados (acumulado: 255)
Página 18: 15 anuncios encontrados (acumulado: 270)
Página 19: 15 anuncios encontrados (acumulado: 285)
Página 20: 15 anuncios enco

In [ ]:
"""======================================================================
FASE 1 (TECNOCASA) BLOQUE 2: EXTRACCIÓN DE LAS URLs NUEVAS (comparando con el CSV existente)
======================================================================"""

nombre_csv = "URLs_recolectadas.csv"

urls_antiguas = set()
if os.path.exists(nombre_csv):
    with open(nombre_csv, "r", encoding="utf-8-sig") as archivo:
        lector = csv.DictReader(archivo)
        for fila in lector:
            urls_antiguas.add(fila["url_detalle"])

urls_nuevas = set(todas_las_urls) - urls_antiguas

print(f"Bloque 2 completado: {len(urls_antiguas)} URLs antiguas leídas, {len(urls_nuevas)} URLs nuevas detectadas.")

Bloque 2 completado: 982 URLs antiguas leídas, 10 URLs nuevas detectadas.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 3: CONTEO — ANTES vs. DESPUÉS DE LA ACTUALIZACIÓN
==================================================================="""

print(f"URLs que había antes de esta ejecución: {len(urls_antiguas)}")
print(f"URLs nuevas encontradas hoy: {len(urls_nuevas)}")
print(f"Total tras la actualización: {len(urls_antiguas) + len(urls_nuevas)}")
print("Bloque 3 completado.")

URLs que había antes de esta ejecución: 982
URLs nuevas encontradas hoy: 10
Total tras la actualización: 992
Bloque 3 completado.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 4: EXPORTAR — AÑADIR LAS URLs NUEVAS AL CSV
==================================================================="""

archivo_existe = os.path.exists(nombre_csv)

with open(nombre_csv, "a", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["url_detalle"])
    if not archivo_existe:
        escritor.writeheader()
    for url in urls_nuevas:
        escritor.writerow({"url_detalle": url})

print(f"Bloque 4 completado. Se han añadido {len(urls_nuevas)} URLs nuevas a {nombre_csv}")

Bloque 4 completado. Se han añadido 10 URLs nuevas a URLs_recolectadas.csv


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 5: DESCARGAR EL CSV ACTUALIZADO
==================================================================="""

files.download(nombre_csv)

print("Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.


In [ ]:
'''================================================
FASE 2 (TECNOCASA): Extracción de datos de cada URL de Tecnocasa
================================================'''

# ATENCIÓN!!!!!! Eliminar antes los archivos de Colab

import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os
from google.colab import files

def extraer_datos_inmueble(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException:
        return None

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return None

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return None

    estate_json = html.unescape(estate_raw)
    estate = json.loads(estate_json)

    # Sub-objetos anidados donde viven planta, año, ascensor, etc.
    features = estate.get("features") or {}
    energy_data = estate.get("energy_data") or {}

    # "tipo" y "distrito" vienen como diccionarios {"id":..., "title":..., "slug":...}
    # nos quedamos solo con el texto legible ("title")
    tipo_dict = estate.get("type") or {}
    distrito_dict = estate.get("district") or {}

    # m2 viene como string numérico limpio ("62.00"), lo convertimos a float directamente
    m2_raw = estate.get("numeric_surface")
    try:
        m2_valor = float(m2_raw) if m2_raw is not None else None
    except ValueError:
        m2_valor = None

    datos = {
        "id": estate.get("id"),
        "detail_url": url,
        "tipo": tipo_dict.get("title"),
        "distrito": distrito_dict.get("title"),
        "barrio": estate.get("quarter"),
        "precio": estate.get("numeric_price"),
        "m2": m2_valor,
        "habitaciones": estate.get("rooms"),
        "banos": estate.get("bathrooms"),
        "planta": features.get("floor"),
        "anio_construccion": features.get("build_year"),
        "anio_reforma": features.get("renovation_year"),
        "categoria": features.get("category"),
        "ascensor": features.get("elevator"),
        "balcones": features.get("balconies"),
        "terrazas": features.get("terraces"),
        "jardin": features.get("garden"),
        "calefaccion": features.get("heating"),
        "clase_energetica": energy_data.get("class"),
        "points_of_interest": json.dumps(estate.get("points_of_interest")),
        "title": estate.get("title"),
        "description": estate.get("description"),
    }

    # Normalizar strings vacíos a None
    for campo in datos:
        if datos[campo] == "":
            datos[campo] = None

    return datos


In [ ]:
'''=================================================
FASE 2 (TECNOCASA) BLOQUE 0: Subida de CSVs previos
=================================================='''

# Sube URLs_recolectadas.csv obligatoriamente.
# Si ya tienes dataset_final.csv y urls_caidas.csv de ejecuciones anteriores, súbelos también.
# Si es la primera vez que corres la Fase 2, no los tendrás: no pasa nada, el código los crea.
subidos = files.upload()

print("Archivos subidos:", list(subidos.keys()))

Saving dataset_final.csv to dataset_final.csv
Saving urls_caidas.csv to urls_caidas.csv
Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Archivos subidos: ['dataset_final.csv', 'urls_caidas.csv', 'URLs_recolectadas.csv']


In [ ]:
'''==============================================================
FASE 2 (TECNOCASA) BLOQUE 1: Calcular URLs pendientes de procesar
==============================================================='''

# Cargar todas las URLs recolectadas en Fase 1
todas_las_urls = set()
with open("URLs_recolectadas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f)
    for fila in lector:
        todas_las_urls.add(fila["url_detalle"])

print(f"Total de URLs recolectadas: {len(todas_las_urls)}")

# Cargar URLs ya procesadas con éxito (si el archivo existe de una ejecución anterior)
urls_ya_procesadas = set()
if os.path.exists("dataset_final.csv"):
    with open("dataset_final.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_ya_procesadas.add(fila["detail_url"])

print(f"URLs ya procesadas con éxito: {len(urls_ya_procesadas)}")

# Cargar URLs que ya sabemos que fallaron (para no reintentarlas)
urls_caidas_previas = set()
if os.path.exists("urls_caidas.csv"):
    with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_caidas_previas.add(fila["url"])

print(f"URLs marcadas como caídas previamente: {len(urls_caidas_previas)}")

# URLs que quedan por procesar en esta ejecución
urls_pendientes = todas_las_urls - urls_ya_procesadas - urls_caidas_previas

print(f"URLs pendientes de extraer en esta ejecución: {len(urls_pendientes)}")

Total de URLs recolectadas: 992
URLs ya procesadas con éxito: 979
URLs marcadas como caídas previamente: 3
URLs pendientes de extraer en esta ejecución: 10


In [ ]:
'''=================================================================
FASE 2 (TECNOCASA) BLOQUE 2: Bucle principal con guardado progresivo
=================================================================='''

# Nombres de los 21 campos que devuelve extraer_datos_inmueble()
campos = ["id", "detail_url", "tipo", "distrito", "barrio", "precio", "m2",
          "habitaciones", "banos", "planta", "anio_construccion", "anio_reforma",
          "categoria", "ascensor", "balcones", "terrazas", "jardin",
          "calefaccion", "clase_energetica", "points_of_interest",
          "title", "description"]

# Si dataset_final.csv no existe todavía, hay que escribir la cabecera primero
escribir_cabecera_dataset = not os.path.exists("dataset_final.csv")
escribir_cabecera_caidas = not os.path.exists("urls_caidas.csv")

contador_ok = 0
contador_fallo = 0

# Abrimos los dos CSVs en modo "a" (append): cada ficha se escribe al momento, no al final
# delimiter=";" para que Excel en español lo abra bien sin descuadrar columnas
with open("dataset_final.csv", "a", newline="", encoding="utf-8-sig") as f_dataset, \
     open("urls_caidas.csv", "a", newline="", encoding="utf-8-sig") as f_caidas:

    escritor_dataset = csv.DictWriter(f_dataset, fieldnames=campos, delimiter=";")
    escritor_caidas = csv.writer(f_caidas, delimiter=";")

    if escribir_cabecera_dataset:
        escritor_dataset.writeheader()
    if escribir_cabecera_caidas:
        escritor_caidas.writerow(["url"])

    for i, url in enumerate(urls_pendientes, start=1):
        datos = extraer_datos_inmueble(url)

        if datos is not None:
            escritor_dataset.writerow(datos)
            contador_ok += 1
        else:
            escritor_caidas.writerow([url])
            contador_fallo += 1

        # Forzamos escritura a disco cada ficha, para no perder nada si Colab se cae
        f_dataset.flush()
        f_caidas.flush()

        if i % 25 == 0:
            print(f"Progreso: {i}/{len(urls_pendientes)} — OK: {contador_ok} — Fallos: {contador_fallo}")

        time.sleep(1.5)  # pausa ética entre peticiones

print(f"\nBucle terminado. Nuevas fichas extraídas: {contador_ok}. Nuevos fallos: {contador_fallo}")


Bucle terminado. Nuevas fichas extraídas: 10. Nuevos fallos: 0


In [ ]:
'''==================================
FASE 2 (TECNOCASA) BLOQUE 3: Resumen
=================================='''

total_dataset = sum(1 for _ in open("dataset_final.csv", encoding="utf-8-sig")) - 1  # -1 por la cabecera
total_caidas = sum(1 for _ in open("urls_caidas.csv", encoding="utf-8-sig")) - 1

print(f"Fichas nuevas en esta ejecución: {contador_ok}")
print(f"URLs caídas nuevas en esta ejecución: {contador_fallo}")
print(f"Total acumulado en dataset_final.csv: {total_dataset}")
print(f"Total acumulado en urls_caidas.csv: {total_caidas}")


# Función de diagnóstico: repite la lógica de extraer_datos_inmueble()
# pero en vez de devolver None, indica el motivo exacto del fallo
def diagnosticar_fallo(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"ERROR DE RED: {e}"

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return "NO SE ENCONTRÓ <estate-show-v2> (¿tipo de anuncio distinto o página vendida con otra plantilla?)"

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return "LA ETIQUETA EXISTE PERO NO TIENE ATRIBUTO :estate"

    return "Debería haber funcionado (revisa manualmente)"


print("\nURLs caídas y motivo del fallo:")
with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f, delimiter=";")
    for fila in lector:
        motivo = diagnosticar_fallo(fila["url"])
        print(f"{fila['url']}\n  → {motivo}\n")
        time.sleep(1.5)

Fichas nuevas en esta ejecución: 10
URLs caídas nuevas en esta ejecución: 0
Total acumulado en dataset_final.csv: 989
Total acumulado en urls_caidas.csv: 3

URLs caídas y motivo del fallo:
https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html



In [ ]:
'''============================================================================
FASE 2 (TECNOCASA) BLOQUE 4: Descarga de dataset_final.csv para subir a GitHub
============================================================================'''

files.download("dataset_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''=========================================================================
FASE 2 (TECNOCASA) BLOQUE 5: Descarga de urls_caidas.csv para subir a GitHub
========================================================================='''

files.download("urls_caidas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''=======================================================
FASE 1 (REDPISO) BLOQUE 0: Imports y configuración inicial
======================================================='''

# ATENCIÓN!!!!! Borrar los archivos de Colab

import requests
from bs4 import BeautifulSoup
import csv
import os
import time
from google.colab import files  # para subir/descargar archivos en Colab

# Cabecera para simular un navegador real (evita bloqueos básicos por user-agent vacío)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

BASE_URL = "https://www.redpiso.es/venta-viviendas/madrid/madrid"
NOMBRE_CSV = "urls_recolectadas_redpiso.csv"
MAX_PAGINAS_SEGURIDAD = 150   # tope de seguridad, muy por encima de las páginas reales
PAUSA_ENTRE_PETICIONES = 1.5  # segundos de espera entre peticiones (cortesía con el servidor)

print("Bloque 0 completado: configuración cargada.")

Bloque 0 completado: configuración cargada.


In [ ]:
'''=========================================================
FASE 1 (REDPISO) BLOQUE 1: Subir el CSV de la sesión anterior (si ya existe)
========================================================='''

# Colab no tiene memoria entre sesiones, así que si ya recolectaste URLs antes,
# aquí las volvemos a cargar. Si es la primera vez, simplemente pulsa "Cancelar"
# en el diálogo de subida y el script empezará desde cero.

urls_previas = set()

print("Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.")
print("Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.")

try:
    subido = files.upload()
    if NOMBRE_CSV in subido:
        with open(NOMBRE_CSV, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter=";")
            for fila in reader:
                urls_previas.add(fila["url_detalle"])
        print(f"CSV cargado: {len(urls_previas)} URLs previas encontradas.")
    else:
        print("No se subió el archivo esperado. Empezamos desde cero.")
except Exception as e:
    print(f"No se subió ningún archivo ({e}). Empezamos desde cero.")

print(f"\nTotal de URLs previas cargadas: {len(urls_previas)}")

Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.
Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.


No se subió el archivo esperado. Empezamos desde cero.

Total de URLs previas cargadas: 0


In [ ]:
'''===========================================================================
FASE 1 (REDPISO) BLOQUE 2: Recorrer todas las páginas de Redpiso y extraer las URLs de anuncio
==========================================================================='''

# Patrón de paginación confirmado:
#   Página 1 -> BASE_URL
#   Página N -> BASE_URL/pagina-N   (N >= 2)
# Cuando una página no tiene anuncios (0 encontrados), hemos llegado al final real.

def url_pagina(n):
    if n == 1:
        return BASE_URL
    return f"{BASE_URL}/pagina-{n}"


def extraer_urls_inmueble(html):
    soup = BeautifulSoup(html, "html.parser")
    urls = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/inmueble/" in href:
            if href.startswith("/"):
                href = "https://www.redpiso.es" + href
            urls.add(href)
    return urls


urls_actuales = set()
pagina = 1

while pagina <= MAX_PAGINAS_SEGURIDAD:
    url = url_pagina(pagina)
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
    except Exception as e:
        print(f"Página {pagina} -> ERROR de conexión: {e}. Reintentando en 5s...")
        time.sleep(5)
        continue

    if r.status_code != 200:
        print(f"Página {pagina} -> status {r.status_code}. Detenemos aquí.")
        break

    urls_pagina = extraer_urls_inmueble(r.text)

    if len(urls_pagina) == 0:
        print(f"Página {pagina} -> 0 anuncios encontrados. Fin de la paginación real.")
        break

    urls_actuales |= urls_pagina
    print(f"Página {pagina} -> {len(urls_pagina)} anuncios | total acumulado: {len(urls_actuales)}")

    pagina += 1
    time.sleep(PAUSA_ENTRE_PETICIONES)

print(f"\nBloque 2 completado. URLs actuales encontradas en la web: {len(urls_actuales)}")

Página 1 -> 12 anuncios | total acumulado: 12
Página 2 -> 12 anuncios | total acumulado: 24
Página 3 -> 12 anuncios | total acumulado: 36
Página 4 -> 12 anuncios | total acumulado: 48
Página 5 -> 12 anuncios | total acumulado: 60
Página 6 -> 12 anuncios | total acumulado: 72
Página 7 -> 12 anuncios | total acumulado: 84
Página 8 -> 12 anuncios | total acumulado: 96
Página 9 -> 12 anuncios | total acumulado: 108
Página 10 -> 12 anuncios | total acumulado: 120
Página 11 -> 12 anuncios | total acumulado: 132
Página 12 -> 12 anuncios | total acumulado: 144
Página 13 -> 12 anuncios | total acumulado: 156
Página 14 -> 12 anuncios | total acumulado: 168
Página 15 -> 12 anuncios | total acumulado: 180
Página 16 -> 12 anuncios | total acumulado: 192
Página 17 -> 12 anuncios | total acumulado: 204
Página 18 -> 12 anuncios | total acumulado: 216
Página 19 -> 12 anuncios | total acumulado: 228
Página 20 -> 12 anuncios | total acumulado: 240
Página 21 -> 12 anuncios | total acumulado: 252
Página 22

In [ ]:
'''=====================================================================
FASE 1 (REDPISO) BLOQUE 3: Comparar lo recolectado ahora con lo que ya teníamos guardado
====================================================================='''

# La resta de conjuntos (set - set) nos da solo las URLs que no existían antes.

urls_nuevas = urls_actuales - urls_previas

print(f"URLs nuevas detectadas en esta sesión: {len(urls_nuevas)}")
if urls_nuevas:
    print("Ejemplo de URL nueva:", list(urls_nuevas)[0])

URLs nuevas detectadas en esta sesión: 778
Ejemplo de URL nueva: https://www.redpiso.es/inmueble/piso-en-venta-en-calle-de-san-narciso-canillejas-san-blas-madrid-RP292026155275


In [ ]:
'''============================================
FASE 1 (REDPISO) BLOQUE 4: Resumen numérico de la actualización
============================================'''

# (Recordatorio: nunca borramos URLs antiguas, aunque el anuncio ya no aparezca
# en la web -> las conservamos como observación histórica, como en Tecnocasa)

total_antes = len(urls_previas)
total_nuevas = len(urls_nuevas)
total_despues = total_antes + total_nuevas

print("=== RESUMEN DE LA ACTUALIZACIÓN ===")
print(f"URLs antes de esta sesión:   {total_antes}")
print(f"URLs nuevas encontradas:     {total_nuevas}")
print(f"URLs totales tras la unión:  {total_despues}")
print(f"(URLs vistas hoy en la web que ya conocíamos: {len(urls_actuales) - total_nuevas})")

=== RESUMEN DE LA ACTUALIZACIÓN ===
URLs antes de esta sesión:   0
URLs nuevas encontradas:     778
URLs totales tras la unión:  778
(URLs vistas hoy en la web que ya conocíamos: 0)


In [ ]:
'''==================================================================
FASE 1 (REDPISO) BLOQUE 5: Guardar el CSV final combinando URLs previas + URLs nuevas
=================================================================='''

# Formato: separador ';' y encoding 'utf-8-sig' (compatibilidad Excel español)

urls_finales = urls_previas | urls_nuevas  # unión de ambos conjuntos, sin duplicados

with open(NOMBRE_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["url_detalle"])
    for url in sorted(urls_finales):
        writer.writerow([url])

print(f"Bloque 5 completado: {NOMBRE_CSV} guardado con {len(urls_finales)} URLs totales.")

Bloque 5 completado: urls_recolectadas_redpiso.csv guardado con 778 URLs totales.


In [ ]:
'''===================================================
FASE 1 (REDPISO) BLOQUE 6: Descargar el CSV actualizado a tu ordenador
==================================================='''

# (Recuerda: descárgalo en su propia celda, sin descargas simultáneas,
# y no lo abras/guardes desde Excel para no cambiar el formato)

files.download(NOMBRE_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>